# Statistics

## Scope, Sources and Counting Units

Figures, tables and numerical observations are computed during book compilation
from `milestone1/data/reported_results.json`. arXiv aggregates come from Hieu's
technical document. Semantic Scholar counts were reconciled from the per-range
metrics, 25,000 crawl records and validation reports dated **11 November 2026**.

The five assigned ranges each contain 5,000 input IDs. A publication is a base
arXiv ID; its revisions are versions. A reference occurrence is counted for its
citing publication. The five ranges describe the assigned corpus.

**Denominators:** arXiv status rates use assigned IDs; Semantic Scholar's
final-status rate uses input publications and includes accepted caches;
arXiv-ID reference coverage uses all local reference occurrences. Overall
rates and means are computed from summed numerators and denominators.

In [ ]:
from pathlib import Path
import sys

# Support execution from the project root or the notebook's own directory.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "myst.yml").is_file() and (p / "milestone1" / "reporting.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from milestone1.reporting import load_results, total, integer, percent, seconds, hms, table

data = load_results()
arxiv = data["arxiv"]["ranges"]
s2 = data["semantic_scholar"]["ranges"]

from milestone1.reporting import analysis, COLORS, chart_style, finish_figure
from matplotlib.ticker import FuncFormatter

plt = chart_style()
labels = [f"Range {r['range']}" for r in arxiv]
positions = list(range(len(labels)))
comma = FuncFormatter(lambda value, _: f"{value:,.0f}")

## How Much of the Assigned Corpus Was Collected?

**Figure 1.** Reported arXiv publication outcomes after Fast Crawl and
Recovery. The left panel covers all assigned IDs; the right expands the
same non-`ok` subset on a different count scale so small categories remain
readable. The labels within the successful segments use assigned IDs as
denominator. Consult the [implementation](implementation.md) for status rules.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained",
                         gridspec_kw={"width_ratios": [1.35, 1]})
ok = [r["ok"] for r in arxiv]
no_tex = [r["no_tex"] for r in arxiv]
errors = [r["error"] for r in arxiv]
axes[0].barh(positions, ok, color=COLORS["teal"], label="ok")
axes[0].barh(positions, no_tex, left=ok, color=COLORS["orange"], label="no_tex")
axes[0].barh(positions, errors, left=[a+b for a,b in zip(ok, no_tex)],
             color=COLORS["red"], label="error")
for i, r in enumerate(arxiv):
    axes[0].text(r["ok"] / 2, i, f"{r['ok']:,} ({100*r['ok']/r['assigned']:.2f}%)",
                 ha="center", va="center", color="white", fontweight="bold")
axes[0].set(xlim=(0, 5000), title="All assigned publications", xlabel="Publication count")
axes[1].barh(positions, no_tex, color=COLORS["orange"])
axes[1].barh(positions, errors, left=no_tex, color=COLORS["red"], hatch="///")
for i, (n, e) in enumerate(zip(no_tex, errors)):
    axes[1].text(n/2, i, str(n), ha="center", va="center")
    axes[1].text(n+e+6, i, f"{e} errors", va="center", fontsize=10)
axes[1].set(xlim=(0, 400), title="Detail: non-ok publications", xlabel="Publication count (different scale)")
for ax in axes:
    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(comma)
    ax.grid(axis="x", alpha=0.18)
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=3, frameon=False)
finish_figure(fig, "statistics-arxiv-outcomes",
              "arXiv outcomes by range, with a separate detail panel for no_tex and error records.")

assigned = total(arxiv, "assigned")
ok_count = total(arxiv, "ok")
non_ok = assigned - ok_count
best = max(arxiv, key=lambda r: r["ok"]/r["assigned"])
lowest = min(arxiv, key=lambda r: r["ok"]/r["assigned"])
analysis(
f"Metadata is reported for {integer(total(arxiv, 'metadata'))} of {integer(assigned)} assigned IDs "
    f"across {integer(total(arxiv, 'batches'))} batches. Publication outcomes are "
    f"{integer(ok_count)} ok ({percent(ok_count, assigned)}), "
    f"{integer(total(arxiv, 'no_tex'))} no_tex ({percent(total(arxiv, 'no_tex'), assigned, 3)}), "
    f"{integer(total(arxiv, 'error'))} error ({percent(total(arxiv, 'error'), assigned, 3)}) and "
    f"{integer(total(arxiv, 'not_found'))} not_found. The ok rate ranges from "
    f"{percent(lowest['ok'], lowest['assigned'])} in Range {lowest['range']} to "
    f"{percent(best['ok'], best['assigned'])} in Range {best['range']}.",
    f"Metadata coverage and retained-source availability describe separate outcomes. "
    f"The no_tex category accounts for {percent(total(arxiv, 'no_tex'), non_ok)} of non-ok publications.",
    "Publication status aggregates version outcomes: any error gives error; otherwise at least "
    "one retained .tex or .bib gives ok. The no_tex category groups PDF, empty content, "
    "no retained files and source HTTP 404/410."
)

## What Does the Semantic Scholar Status Rate Measure?

**Figure 2.** Outcomes in the 11 November reruns. The left panel separates
accepted caches, fresh `ok` outcomes and fresh `not_indexed` outcomes. The right
panel expands the fresh subset on its own count scale. There are zero final
`failed` and `invalid_id` records.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained",
                         gridspec_kw={"width_ratios": [1.35, 1]})
cached = [r["cache_hits"] for r in s2]
fresh_ok = [r["ok"] - r["cache_hits"] for r in s2]
missing = [r["not_indexed"] for r in s2]
axes[0].barh(positions, cached, color=COLORS["blue"], label="Accepted cache")
axes[0].barh(positions, fresh_ok, left=cached, color=COLORS["teal"], label="Fresh ok")
axes[0].barh(positions, missing, left=[a+b for a,b in zip(cached, fresh_ok)],
             color=COLORS["red"], label="Fresh not_indexed")
for i, r in enumerate(s2):
    axes[0].text(r["cache_hits"]/2, i, f"{r['cache_hits']:,} cached",
                 ha="center", va="center", color="white", fontweight="bold")
axes[0].set(xlim=(0, 5000), title="All input publications", xlabel="Publication count")
axes[1].barh(positions, fresh_ok, color=COLORS["teal"])
axes[1].barh(positions, missing, left=fresh_ok, color=COLORS["red"], hatch="///")
for i, (n, m) in enumerate(zip(fresh_ok, missing)):
    if n:
        axes[1].text(n/2, i, str(n), ha="center", va="center", color="white", fontweight="bold")
    axes[1].text(n+m+0.25, i, f"{m} not indexed", va="center", fontsize=9)
axes[1].set(xlim=(0, 9), title="Detail: fresh lookups only", xlabel="Publication count (different scale)")
for ax in axes:
    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(comma)
    ax.grid(axis="x", alpha=0.18)
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=3, frameon=False)
finish_figure(fig, "statistics-semantic-scholar-outcomes",
              "Semantic Scholar rerun outcomes are dominated by caches; fresh lookups are shown separately.")

inputs = total(s2, "input_papers")
accepted = total(s2, "ok")
cached_count = total(s2, "cache_hits")
fresh_count = inputs - cached_count - total(s2, "invalid_id")
analysis(
f"The final publication-level ok rate is {integer(accepted)} / {integer(inputs)} = "
    f"{percent(accepted, inputs, 3)} including accepted caches. There are {integer(cached_count)} cache hits "
    f"and {integer(fresh_count)} fresh lookups: {integer(accepted-cached_count)} ok and "
    f"{integer(total(s2, 'not_indexed'))} not_indexed (HTTP 404).",
    "Accepted saved reference lists account for the ok outcomes in these reruns. "
    "Semantic Scholar indexing and arXiv source availability are separate outcomes.",
    f"The {integer(fresh_count)} fresh publication lookups generated {integer(total(s2, 'http_requests'))} "
    f"HTTP attempts, including {integer(total(s2, 'http_retries'))} retries and "
    f"{integer(total(s2, 'http_429'))} HTTP 429 responses. Publication counts and HTTP-attempt counts "
    "therefore describe different units."
)

## How Much of the Reference Pool Carries an arXiv ID?

**Figure 3.** Reference occurrences retained locally, split by the presence
of a usable arXiv ID. Counts appear on the left; percentages of each range's
local references appear on the right. The dashed line is the overall share
weighted by reference counts. Both panels describe identifier coverage.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained",
                         gridspec_kw={"width_ratios": [1.35, 1]})
with_id = [r["refs_with_arxiv"] for r in s2]
without_id = [r["refs_total"]-r["refs_with_arxiv"] for r in s2]
axes[0].barh(positions, with_id, color=COLORS["teal"], label="With arXiv ID")
axes[0].barh(positions, without_id, left=with_id, color=COLORS["gray"], label="Without arXiv ID")
for i, r in enumerate(s2):
    axes[0].text(r["refs_total"]+2500, i, integer(r["refs_total"]), va="center", fontsize=9)
axes[0].set(xlim=(0, 230000), title="Local reference counts", xlabel="Reference occurrences")
axes[0].xaxis.set_major_formatter(comma)
shares = [100*r["refs_with_arxiv"]/r["refs_total"] for r in s2]
bars = axes[1].barh(positions, shares, color=COLORS["teal"])
axes[1].bar_label(bars, labels=[f"{v:.2f}%" for v in shares], padding=4)
overall_share = 100*total(s2, "refs_with_arxiv")/total(s2, "refs_total")
axes[1].axvline(overall_share, color=COLORS["purple"], linestyle="--", linewidth=1.6,
                label=f"Overall: {overall_share:.2f}%")
axes[1].set(xlim=(0, 100), title="arXiv ID coverage", xlabel="Share of local references (%)")
axes[1].legend(loc="lower right", frameon=False, fontsize=9)
for ax in axes:
    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.18)
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=2, frameon=False)
finish_figure(fig, "statistics-reference-coverage",
              "Reference counts and arXiv-ID coverage by range, with a weighted overall coverage of 49.83 percent.")

all_refs = total(s2, "refs_total")
arxiv_refs = total(s2, "refs_with_arxiv")
largest = max(s2, key=lambda r: r["refs_total"])
highest_share = max(s2, key=lambda r: r["refs_with_arxiv"]/r["refs_total"])
analysis(
f"The reconciled exports contain {integer(all_refs)} local reference occurrences. "
    f"Of these, {integer(arxiv_refs)} ({percent(arxiv_refs, all_refs)}) carry arXiv IDs and "
    f"{integer(all_refs-arxiv_refs)} ({percent(all_refs-arxiv_refs, all_refs)}) do not. "
    f"Range {largest['range']} has the largest count ({integer(largest['refs_total'])}); "
    f"Range {highest_share['range']} has the highest arXiv-bearing share "
    f"({percent(highest_share['refs_with_arxiv'], highest_share['refs_total'])}).",
    "Identifier coverage determines the size of the arXiv-keyed matching pool. References "
    "without an arXiv ID remain in the local corpus with their collected metadata.",
    "Counts describe citation occurrences tied to citing publications. The same cited work "
    "can appear in several publications, and its arXiv ID can lie outside the assigned corpus."
)

## How Many References Are Associated with an Input Publication?

**Figure 4.** Mean local reference counts and accepted zero-reference rates.
Each range uses every input publication as denominator, including inputs
without an indexed result. The empty-record numerator counts records accepted
as empty by the implementation. These panels compare different quantities;
they do not show individual-paper distributions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained")
means = [r["refs_total"]/r["input_papers"] for r in s2]
empty = [100*r["zero_refs"]/r["input_papers"] for r in s2]
bars = axes[0].barh(positions, means, color=COLORS["blue"])
axes[0].bar_label(bars, labels=[f"{v:.2f}" for v in means], padding=4)
overall_mean = total(s2, "refs_total")/total(s2, "input_papers")
axes[0].axvline(overall_mean, color=COLORS["purple"], linestyle="--",
                label=f"Overall: {overall_mean:.2f}")
axes[0].set(xlim=(0, 46), title="Mean local reference count", xlabel="Reference occurrences / input publication")
axes[0].legend(loc="lower left", frameon=False, fontsize=9)
bars = axes[1].barh(positions, empty, color=COLORS["orange"])
axes[1].bar_label(bars, labels=[f"{v:.2f}% ({r['zero_refs']:,})" for v,r in zip(empty,s2)], padding=4)
axes[1].set(xlim=(0, 15), title="Accepted zero-reference records", xlabel="Share of input publications (%)")
for ax in axes:
    ax.set_yticks(positions, labels)
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.18)
finish_figure(fig, "statistics-references-per-publication",
              "Mean reference counts and zero-reference rates across the five ranges, with all inputs as denominators.")

inputs = total(s2, "input_papers")
all_refs = total(s2, "refs_total")
accepted = total(s2, "ok")
highest_mean = max(s2, key=lambda r: r["refs_total"]/r["input_papers"])
lowest_mean = min(s2, key=lambda r: r["refs_total"]/r["input_papers"])
analysis(
f"The pooled mean is {all_refs/inputs:.2f} references per input publication; using only "
    f"{integer(accepted)} ok records gives {all_refs/accepted:.2f}. Range {highest_mean['range']} "
    f"has the highest mean ({highest_mean['refs_total']/highest_mean['input_papers']:.2f}) and "
    f"Range {lowest_mean['range']} the lowest ({lowest_mean['refs_total']/lowest_mean['input_papers']:.2f}). "
    f"There are {integer(total(s2, 'zero_refs'))} saved zero-reference records "
    f"({percent(total(s2, 'zero_refs'), inputs)} of inputs).",
    "The mean summarizes the collected reference workload. Including every input describes "
    "the assigned corpus; using only ok records describes the accepted-record subset.",
    "The empty-record rate counts saved empty reference lists and uses all input publications "
    "as denominator. Each range mean is a summary of that range's stored reference occurrences."
)

## Data Quality and Supporting Counts

**Table 1.** Exact arXiv publication outcomes underlying Figure 1. These
document-reported counts are generated at compilation for inspection.

In [ ]:
keys = ("assigned", "metadata", "ok", "error", "no_tex", "not_found")
rows = [[f"Range {r['range']}", *[integer(r[k]) for k in keys],
         percent(r["ok"], r["assigned"])] for r in arxiv]
rows.append(["Overall", *[integer(total(arxiv, k)) for k in keys],
             percent(total(arxiv, "ok"), total(arxiv, "assigned"))])
table(["Range", "Assigned IDs", "Metadata", "ok", "error", "no_tex", "not_found", "ok / assigned"], rows)

**Table 2.** Exact reference counts, identifier shares and means underlying
Figures 3 and 4. The mean uses all input publications.

In [ ]:
rows = []
for r in s2:
    rows.append([f"Range {r['range']}", integer(r["refs_total"]), integer(r["refs_with_arxiv"]),
                 integer(r["refs_total"] - r["refs_with_arxiv"]),
                 percent(r["refs_with_arxiv"], r["refs_total"]),
                 f"{r['refs_total'] / r['input_papers']:.2f}"])
all_refs, arxiv_refs, inputs = total(s2, "refs_total"), total(s2, "refs_with_arxiv"), total(s2, "input_papers")
rows.append(["Overall", integer(all_refs), integer(arxiv_refs), integer(all_refs - arxiv_refs),
             percent(arxiv_refs, all_refs), f"{all_refs / inputs:.2f}"])
table(["Range", "All local refs", "With arXiv ID", "Without arXiv ID", "arXiv share", "Refs / input paper"], rows)

**Table 3.** Metadata warnings in the updated validation exports. Every range
has 5,000 paper rows and zero validation issues. The number of reference rows
matches the arXiv-bearing count in Table 2. Ranges 4 and 5 each retain one row
with an empty `ref_authors` list under the documented missing-author policy.

In [ ]:
details = data["semantic_scholar"]["validation_details"]
table(["Range", "Table", "Reported index", "Field", "Finding", "Classification"],
      [[f"Range {r['range']}", r["table"], str(r["reported_index"]), r["field"],
        r["problem"], r["classification"]] for r in details])

The validator allows empty author lists and preserves the two rows with
metadata warnings. The reports record unchanged reference counts before and
after identifier-based deduplication for each range.

`ref_submission_date` stores Semantic Scholar publication date/year. The
[method](semantic_scholar.md) describes the field mapping and the citing/cited
ID relationship.